In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
username = "root"
database = "company_db"
password = "jessepinkman"
host = "localhost"

In [3]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

In [4]:
llm = ChatOllama(model="llama3")

In [5]:
schema = """
Table: employees
Columns: employee_id, employee_name, age, gender, department_id, salary, joining_date, city

Table: departments
Columns: department_id, department_name

Table: customers
Columns: customer_id, customer_name, gender, age, city, phone_number, email, registration_date

Table: products
Columns: product_id, product_name, category, brand, price, stock_quantity

Table: orders
Columns: order_id, customer_id, product_id, employee_id, quantity, order_amount, order_date, payment_method, order_status

Table: sales
Columns: sale_id, employee_id, product_name, sale_amount, sale_date, region
"""

In [6]:
class AgentState(TypedDict):
    user_question: str
    generated_sql: str
    dataframe: pd.DataFrame
    needs_chart: str

In [7]:
def generate_sql(state: AgentState):

    user_question = state.get("user_question", "")

    print("User Question: ", user_question)

    prompt = f"""
    You are an expert MySQL query generator.

    Database Schema:
    {schema}

    User Question:
    {user_question}

    Important Rules:
    1. Return exactly ONE MySQL query
    2. Do not explain anything
    3. Do not return multiple queries
    4. Do not return markdown
    5. Do not add examples
    6. Only use the provided tables and columns
    7. Return only plain SQL text   
    """

    response = llm.invoke(prompt)

    generated_sql = response.content.strip()

    print("Generated SQL:")
    print(generated_sql)

    state["generated_sql"] = generated_sql

    return state

In [8]:
def execute_sql(state: AgentState):

    generated_sql = state.get("generated_sql", "")

    df = pd.read_sql(generated_sql, engine)

    print("Result:")
    print(df.head())

    state["dataframe"] = df

    return state

In [9]:
def decide_chart(state: AgentState):

    user_question = state.get("user_question", "")
    df = state.get("dataframe", pd.DataFrame())

    chart_prompt = f"""
    User Question: {user_question}

    Data Columns: {list(df.columns)}

    Should this result be shown as a chart?

    Only answer YES or NO.
    """

    response = llm.invoke(chart_prompt)

    needs_chart = response.content.strip().upper()

    print("Needs Chart:", needs_chart)

    state["needs_chart"] = needs_chart

    return state

In [10]:
def generate_chart(state: AgentState):

    df = state["dataframe"]
    user_question = state["user_question"]

    if len(df.columns) >= 2:

        x_column = df.columns[0]
        y_column = df.columns[1]

        df = df.head(5).copy()

        df[y_column] = pd.to_numeric(df[y_column], errors="coerce")
        df = df.dropna()

        plt.figure(figsize=(6,4))
        plt.bar(df[x_column].astype(str), df[y_column])

        plt.title(user_question)
        plt.xlabel(x_column)
        plt.ylabel(y_column)

        plt.xticks(rotation=30)
        plt.tight_layout()

        plt.show()
        plt.close()

    return state

In [11]:
def route_chart(state: AgentState):

    if state["needs_chart"] == "YES":
        return "chart"
    else:
        return "end"

In [12]:
workflow = StateGraph(AgentState)

workflow.add_node("generate_sql", generate_sql)
workflow.add_node("execute_sql", execute_sql)
workflow.add_node("decide_chart", decide_chart)
workflow.add_node("generate_chart", generate_chart)

workflow.set_entry_point("generate_sql")

workflow.add_edge("generate_sql", "execute_sql")
workflow.add_edge("execute_sql", "decide_chart")

workflow.add_conditional_edges(
    "decide_chart",
    route_chart,
    {
        "chart": "generate_chart",
        "end": END
    }
)

workflow.add_edge("generate_chart", END)

In [13]:
app = workflow.compile()

In [14]:
result = app.invoke(
    {
        "user_question": "Show total sales by region"
    }
)

User Question:  Show total sales by region
Generated SQL:
SELECT region, SUM(sale_amount) AS total_sales 
FROM sales 
GROUP BY region;
Result:
  region  total_sales
0   West     446200.0
1  North     374100.0
2  South     342600.0
3   East     106750.0
Needs Chart: NO
